# Data cleaning & manipulation .ipynb

## Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.data_cleaning_and_manipulations import impute_by_agency_line_hour

## Imports

In [11]:
df = pd.read_csv(r'../data/model_datasets/train_df.csv',encoding='utf-8-sig')
df.head()

,date,day,full_hour,line_num,line_name,route_id,route_mkt,direction,alternative,agency_name,...,duration_difference_min,speed_kmh_planned,speed_kmh_actual,rainfall_mm,length_in_buffer_m,route_length,perc_within_pt_route,curvity,Total_Passengers,Avg_Passengers_Per_Bus
0,2024-04-07,Sunday,12,9.0,מסוף הטייסים-תל אביב יפו<->קריית חינוך-תל אביב...,2294,17009,2,0,Dan,...,32.533333,21.925313,11.4,0.0,2409.179192,15883.671217,15.17,1.600497,121.0,24.200000
1,2024-04-10,Wednesday,17,10.0,מסוף אזורי חן-תל אביב יפו<->חניון אגד/דן שומרו...,2296,18010,1,0,Dan,...,35.000000,19.000000,12.4,0.0,3873.941863,21117.321745,18.34,1.439460,322.0,64.400000
2,2024-04-08,Monday,23,201.0,יצחק אלחנן/הכובשים-תל אביב יפו<->ת. מרכזית רחו...,3354,10201,1,#,Egged,...,9.000000,22.800000,20.1,0.0,14020.628337,26839.569509,52.24,1.368459,38.0,9.500000
3,2024-04-09,Tuesday,20,172.0,מסוף רדינג/רציפים-תל אביב יפו<->חניון תמנע-חול...,9808,12172,1,0,Dan,...,16.000000,17.500000,13.8,0.0,5734.977901,18894.935802,30.35,2.305428,268.0,33.500000
4,2024-04-09,Tuesday,21,501.0,ת.מרכזית תל אביב קומה 6/רציפים-תל אביב יפו<->מ...,5193,11501,2,#,Metropolin,...,7.000000,23.300000,20.8,0.0,12542.965847,24754.731868,50.67,1.458932,100.0,33.333333


## 1. Data types

In [ ]:
from src.data_preparation import fix_data_types

df = fix_data_types(df)
print(df.dtypes)

### 1.1. Date & time

In [ ]:
# Date
df['date'] = pd.to_datetime(df['date'])

# חלץ רק את השעה מהעמודות
df['departure_time_planned'] = pd.to_datetime(
    df['date'].dt.strftime('%Y-%m-%d') + ' ' + 
    pd.to_datetime(df['departure_time_planned'], format='mixed').dt.strftime('%H:%M:%S'),
    format='%Y-%m-%d %H:%M:%S'
)

df['arrival_time_planned'] = pd.to_datetime(
    df['date'].dt.strftime('%Y-%m-%d') + ' ' + 
    pd.to_datetime(df['arrival_time_planned'], format='mixed').dt.strftime('%H:%M:%S'),
    format='%Y-%m-%d %H:%M:%S'
)

print(df[['date', 'departure_time_planned', 'arrival_time_planned']].head())

### 1.2. Numeric

In [ ]:
## Hour
df = df.rename(columns={'hour_rounded': 'full_hour'})
print(df['full_hour'].dtype)
## Line_num
df['line_num'] = [pd.to](http://pd.to)_numeric(df['line_num'], errors='coerce').astype('Int64')
print(df['line_num'].dtype)

### 1.3. Categorial

In [ ]:
# day - categorical with order
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df['day'] = pd.Categorical(df['day'], categories=day_order, ordered=True)
print(df['day'].dtype)
str_cols = ['line_name', 'alternative', 'agency_name', 'origin_city', 
            'origin_station', 'destination_city', 'destination_station', 'route_type']
df[str_cols] = df[str_cols].astype(str)
df['route_type'] = df['route_type'].str.strip()

## 2. Missing values handling

In [ ]:
from src.data_preparation import handle_missing_values

df = handle_missing_values(df, cols=['Total_Passengers'])

### 2.2. Passengrs columns

In [ ]:
for col in ['Total_Passengers']:
    df[col] = df.groupby(['route_id', 'direction', 'day', 'full_hour'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    print(f"{col} missing after fill: {df[col].isna().sum()}")


# שלב 2 - groupby פחות ספציפי
for col in ['Total_Passengers']:
    df[col] = df.groupby(['route_id', 'direction', 'day'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    print(f"{col} missing after step 2: {df[col].isna().sum()}")

# שלב 3 - אם עדיין חסר, לפי route_id בלבד
for col in ['Total_Passengers']:
    df[col] = df.groupby(['route_id'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    print(f"{col} missing after step 3: {df[col].isna().sum()}")

# שלב 4 - אם עדיין חסר, מלא עם חציון כללי
for col in ['Total_Passengers']:
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} missing after step 4: {df[col].isna().sum()}")


missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(by='missing_percent', ascending=False)

missing_summary

### 2.3 Duration columns

In [ ]:
### Impute the remaining nulls of the duration and speed based on agency, line and hor average

df = impute_by_agency_line_hour(df, 'duration_min_actual')
df = impute_by_agency_line_hour(df, 'duration_difference_min')
df = impute_by_agency_line_hour(df, 'speed_kmh_actual')

### 2.3. Spatial columns

In [ ]:
cols = [
    "curvity",
    "perc_within_pt_route",
    "route_length",
    "length_in_buffer_m"
]

for col in cols:
    df[col] = df[col].fillna(df[col].mean())

### 2.4. Missing values summary

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(by='missing_percent', ascending=False)

missing_summary

## 3. Outliers handling

In [ ]:
from src.data_preparation import fix_speed_and_duration

df = fix_speed_and_duration(df)

#### 3.1.1. Very high speed

In [ ]:
mask_high = df['speed_kmh_planned'] > 100
print(f"Rows with speed > 100: {mask_high.sum()}")

df.loc[mask_high, 'speed_kmh_planned'] = df.loc[mask_high, 'speed_kmh_planned'] / 1000

print(df['speed_kmh_planned'].describe())

#### 3.1.2. Very low speed

In [ ]:
df[df['speed_kmh_planned'] < 0][['date', 'route_length', 'duration_min_planned', 'departure_time_planned', 'arrival_time_planned', 'speed_kmh_planned']].head(10)

## It looks like it happend because the ries ends in the next day

In [ ]:
df['departure_time_planned'] = pd.to_datetime(df['departure_time_planned'].astype(str), format='mixed')
df['arrival_time_planned'] = pd.to_datetime(df['arrival_time_planned'].astype(str), format='mixed')

# זהה נסיעות שמסתיימות אחרי חצות
mask_midnight = df['arrival_time_planned'] < df['departure_time_planned']
print(f"Trips ending after midnight: {mask_midnight.sum()}")

# תקן את duration_min_planned
df.loc[mask_midnight, 'duration_min_planned'] = (
    (df.loc[mask_midnight, 'arrival_time_planned'] + pd.Timedelta(days=1)) - 
    df.loc[mask_midnight, 'departure_time_planned']
).dt.total_seconds() / 60

# תקן את speed_kmh_planned
df.loc[mask_midnight, 'speed_kmh_planned'] = (
    (df.loc[mask_midnight, 'route_length'] / 1000) / 
    (df.loc[mask_midnight, 'duration_min_planned'] / 60)
)

print(df[mask_midnight][['departure_time_planned', 'arrival_time_planned', 
                          'duration_min_planned', 'speed_kmh_planned']].head())

In [ ]:
df[df['speed_kmh_actual'] > 500][['route_length', 'duration_min_planned', 'departure_time_planned', 'arrival_time_planned', 'speed_kmh_planned', 'duration_min_actual', 'speed_kmh_actual']].head(10)

In [ ]:
before = len(df)

df = df[(df['duration_difference_min'] >= -120) & 
        (df['duration_difference_min'] <= 120)]

after = len(df)
print(f"Rows removed: {before - after:,} ({(before-after)/before*100:.2f}%)")
print(f"Rows remaining: {after:,}")
print(df['duration_difference_min'].describe().round(2))

#### 3.1.3. Check speed box plot

## 4. Feature selection

In [ ]:
from src.data_preparation import drop_unnecessary_columns

df = drop_unnecessary_columns(df)

In [ ]:
df = df.drop(columns=['route_dir_alt_day_hr', 'line_num_agency_alter_dir', 'SIRI_id', 'gtfs_ride_id', 'gtfs_route_id' ], errors='ignore')
df.dtypes

## 5. Encoding

In [ ]:
from src.data_preparation import encode_categorical_columns

df = encode_categorical_columns(df)

### 5.1. Ordinal

In [ ]:
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

day_mapping = {day: i for i, day in enumerate(day_order)}
df['day_encoded'] = df['day'].map(day_mapping)

print(df[['day', 'day_encoded']].drop_duplicates().sort_values('day_encoded'))

### 5.2. Get dummies

In [ ]:
alternative_dummies = pd.get_dummies(df['alternative'], prefix='alternative')

df = pd.concat([df, alternative_dummies], axis=1)

print(f"New columns added: {alternative_dummies.columns.tolist()}")
print(df[['alternative'] + alternative_dummies.columns.tolist()].drop_duplicates().sort_values('alternative'))

### 5.3. Target

In [ ]:


target_cols = ['agency_name', 'origin_city', 'destination_city', 
               'origin_station', 'destination_station']

te = TargetEncoder()
df[[f'{col}_encoded' for col in target_cols]] = te.fit_transform(
    df[target_cols], 
    df['duration_difference_min']
)

print("New encoded columns:")
for col in target_cols:
    print(f"\n{col} vs {col}_encoded:")
    print(df[[col, f'{col}_encoded']].drop_duplicates().head(5).round(2))

## 6. Feature engieneering

In [3]:
### Add circular route flag
from src.data_preparation import add_circular_route_flag
df = add_circular_route_flag(
    df,
    routes_linestring_path="../data/interim_data/routes_linestrings.csv",
    route_id_col="route_id"
)

In [9]:
df['urban'] = df['route_length'] <=25000

In [10]:
### Calculating the percetnage of the distance inside the public transport path from the entire route length
df['perc_within_pt_route'] = round(df['length_in_buffer_m']/df['route_length']*100,2)

In [ ]:
peak_hours = [7, 8, 9, 14, 15, 16, 17]

df['perc_within_pt_route_peak'] = df.apply(
    lambda row: row['perc_within_pt_route'] if row['full_hour'] in peak_hours else 0,
    axis=1
)

df['is_peak_hour'] = df['full_hour'].isin(peak_hours).astype(int)

## 7. Manipulation pipelines